# 商品分析对象与精确映射初始化

每类 5 个，共 15 个标准商品。全量采集保持不变，仅分析视图筛选。
商品官网 ID 按超市分别维护；精确名称/规格/分类匹配用于学习来源 SKU，再通过已确认 SKU 映射。
pending_review 暂缓映射，不能通过别名回退绕过审核。清洗原值保留在 raw_specification/raw_reported_unit。
不使用模糊名称匹配，不将测试报价写入正式事实表。肉类比较需要进一步核实。
此初始化保留已有门店、商品和手工确认的映射；重复运行不覆盖已维护的选择标记。


In [ ]:
# Add nullable metadata columns without changing existing values.
MIGRATIONS = {'workspace.default.bronze_prices': {'source_product_id': 'STRING',
                                     'product_url': 'STRING',
                                     'member_price': 'STRING',
                                     'member_unit_price': 'STRING',
                                     'was_price': 'STRING',
                                     'store_verified': 'STRING'},
 'workspace.default.silver_dim_product': {'is_analysis_selected': 'BOOLEAN',
                                          'analysis_rank': 'INT',
                                          'selection_notes': 'STRING',
                                          'comparison_ready': 'BOOLEAN'},
 'workspace.default.silver_fact_product_price': {'source_product_id': 'STRING',
                                                 'product_url': 'STRING',
                                                 'member_price': 'DECIMAL(12,4)',
                                                 'member_unit_price': 'DECIMAL(12,4)',
                                                 'was_price': 'DECIMAL(12,4)',
                                                 'store_verification_status': 'STRING',
                                                 'raw_specification': 'STRING',
                                                 'raw_reported_unit': 'STRING',
                                                 'specification_quality_status': 'STRING',
                                                 'unit_quality_status': 'STRING',
                                                 'normalized_unit': 'STRING',
                                                 'normalized_unit_price': 'DECIMAL(16,6)',
                                                 'normalized_member_unit_price': 'DECIMAL(16,6)'}}
for table,columns in MIGRATIONS.items():
    if not spark.catalog.tableExists(table):
        raise ValueError(f"Run setup_price_model / initial ingestion first: {table}")
    current=set(spark.table(table).columns)
    missing=[f"{name} {kind}" for name,kind in columns.items() if name not in current]
    if missing:spark.sql(f"ALTER TABLE {table} ADD COLUMNS ({', '.join(missing)})")


## 01 · 维护商品 SKU 映射

按品牌及官网 SKU 维护映射；已确认的映射与待审核状态分别处理。


In [ ]:
spark.sql(r"""
CREATE TABLE IF NOT EXISTS workspace.default.silver_dim_product_source (source STRING NOT NULL, source_product_id STRING NOT NULL, product_id STRING REFERENCES workspace.default.silver_dim_product(product_id), original_name STRING, original_specification STRING, category_id INT, product_url STRING, mapping_status STRING, mapping_note STRING, PRIMARY KEY (source,source_product_id)) USING DELTA COMMENT 'Source SKU registry. Only approved/exact_reference mappings resolve a product; pending stays unmapped.'
""")


## 02 · silver_dim_product · UPDATE

执行下方 SQL，按 Run all 的顺序处理。


In [ ]:
spark.sql(r"""
UPDATE workspace.default.silver_dim_product
   SET is_analysis_selected=false
 WHERE is_analysis_selected IS NULL
""")


## 03 · silver_dim_product · MERGE

执行下方 SQL，按 Run all 的顺序处理。


In [ ]:
spark.sql(r"""
MERGE INTO workspace.default.silver_dim_product t USING (
        SELECT *
          FROM
        VALUES ('0001',true,1,'Match brand, egg grade, farming method and pack size.',true), ('0003',true,2,'Match brand, egg grade, farming method and pack size.',true), ('0011',true,3,'Match brand, egg grade, farming method and pack size.',true), ('0014',true,4,'Match brand, egg grade, farming method and pack size.',true), ('0015',true,5,'Match brand, egg grade, farming method and pack size.',true), ('0019',true,1,'Match Meadow Fresh product line, fat type and bottle volume.',true), ('0020',true,2,'Match Meadow Fresh product line, fat type and bottle volume.',true), ('0021',true,3,'Match Meadow Fresh product line, fat type and bottle volume.',true), ('0022',true,4,'Match Meadow Fresh product line, fat type and bottle volume.',true), ('0023',true,5,'Match Meadow Fresh product line, fat type and bottle volume.',true), ('0042',true,1,'Review NZ origin, cut, fresh/frozen, bone and fat content, plus per-kg vs pack pricing before comparison.',false), ('0043',true,2,'Review NZ origin, cut, fresh/frozen, bone and fat content, plus per-kg vs pack pricing before comparison.',false), ('0045',true,3,'Review NZ origin, cut, fresh/frozen, bone and fat content, plus per-kg vs pack pricing before comparison.',false), ('0047',true,4,'Review NZ origin, cut, fresh/frozen, bone and fat content, plus per-kg vs pack pricing before comparison.',false), ('0048',true,5,'Review NZ origin, cut, fresh/frozen, bone and fat content, plus per-kg vs pack pricing before comparison.',false) AS v(product_id,is_analysis_selected,analysis_rank,selection_notes,comparison_ready)
       ) s
    ON t.product_id=s.product_id WHEN MATCHED
   AND t.analysis_rank IS NULL THEN UPDATE
   SET t.is_analysis_selected=s.is_analysis_selected,
       t.analysis_rank=s.analysis_rank,
       t.selection_notes=s.selection_notes,
       t.comparison_ready=s.comparison_ready
""")


## 04 · silver_dim_product_alias · MERGE

执行下方 SQL，按 Run all 的顺序处理。


In [ ]:
spark.sql(r"""
MERGE INTO workspace.default.silver_dim_product_alias t USING (
        SELECT *
          FROM
        VALUES ('woolworths','Henergy Eggs Cage Free Size 6 12 Pack','12pk','0015'), ('woolworths','Farmer Brown Eggs Cage Free Barn Size 6 12 Pack','12pk','0001') AS v(source,original_name,original_specification,product_id)
       ) s
    ON t.source=s.source
   AND t.original_name=s.original_name
   AND t.original_specification=s.original_specification WHEN NOT MATCHED THEN INSERT *
""")


## 05 · 维护商品 SKU 映射

按品牌及官网 SKU 维护映射；已确认的映射与待审核状态分别处理。


In [ ]:
spark.sql(r"""
MERGE INTO workspace.default.silver_dim_product_source t USING (
        SELECT *
          FROM
        VALUES ('newworld','product-5269558-EA-000','0012','Woodland Free Range Size 6 Eggs','18pk',1,NULL,'exact_reference','Observed verified-store test SKU; exact reference match only.'), ('newworld','product-5240876-EA-000',NULL,'Farmer Brown Fresh Colony Caged Size 7 Eggs','12pk',1,NULL,'pending','Observed verified-store test SKU; exact reference match only.'), ('newworld','product-5101827-EA-000',NULL,'Natural Free-Range Co Natural Free-Range Eggs','12pk',1,NULL,'pending','Observed verified-store test SKU; exact reference match only.'), ('newworld','product-5021173-EA-000',NULL,'Zeagold Egg White Natural Protein','980ml',1,NULL,'pending','Observed verified-store test SKU; exact reference match only.'), ('newworld','product-5309533-EA-000',NULL,'Farmer Brown Fresh Colony Caged Size 8 Eggs','10pk',1,NULL,'pending','Observed verified-store test SKU; exact reference match only.'), ('newworld','product-5023573-EA-000','0014','Woodland Free Range Size 7 Eggs','10pk',1,NULL,'exact_reference','Observed verified-store test SKU; exact reference match only.'), ('newworld','product-5256998-EA-000','0001','Farmer Brown Cage-Free Barn Size 6 Eggs','12pk',1,NULL,'exact_reference','Observed verified-store test SKU; exact reference match only.'), ('newworld','product-5023568-EA-000','0011','Woodland Free Range Size 6 Eggs','10pk',1,NULL,'exact_reference','Observed verified-store test SKU; exact reference match only.'), ('newworld','product-5314652-EA-000',NULL,'Farmer Brown Fresh Colony Caged Size 6 Eggs','18pk',1,NULL,'pending','Observed verified-store test SKU; exact reference match only.'), ('newworld','product-5303468-EA-000',NULL,'Bowalley Free Range Mixed Grade Eggs','10pk',1,NULL,'pending','Observed verified-store test SKU; exact reference match only.'), ('newworld','product-5040640-EA-000',NULL,'Woodland Free Range Grade 8 Eggs','6pk',1,NULL,'pending','Observed verified-store test SKU; exact reference match only.'), ('paknsave','product-5240876-EA-000',NULL,'Farmer Brown Fresh Colony Caged Size 7 Eggs','12pk',1,NULL,'pending','Observed verified-store test SKU; exact reference match only.'), ('paknsave','product-5310558-EA-000',NULL,'Sure As Eggs Mixed Grade Colony Caged Eggs','10pk',1,NULL,'pending','Observed verified-store test SKU; exact reference match only.'), ('paknsave','product-5023573-EA-000','0014','Woodland Free Range Size 7 Eggs','10pk',1,NULL,'exact_reference','Observed verified-store test SKU; exact reference match only.'), ('paknsave','product-5023568-EA-000','0011','Woodland Free Range Size 6 Eggs','10pk',1,NULL,'exact_reference','Observed verified-store test SKU; exact reference match only.'), ('paknsave','product-5256998-EA-000','0001','Farmer Brown Cage-Free Barn Size 6 Eggs','12pk',1,NULL,'exact_reference','Observed verified-store test SKU; exact reference match only.'), ('woolworths','352492','0015','Henergy Eggs Cage Free Size 6 12 Pack','12pk',1,'https://www.woolworths.co.nz/shop/product-details/352492/henergy-eggs-cage-free-size-6','exact_reference','Observed verified-store test SKU; exact reference match only.'), ('woolworths','422870',NULL,'Woolworths Eggs Barn Mixed Grade 20 Pack','20pk',1,'https://www.woolworths.co.nz/shop/product-details/422870/woolworths-eggs-barn-mixed-grade','pending','Observed verified-store test SKU; exact reference match only.'), ('woolworths','52216','0001','Farmer Brown Eggs Cage Free Barn Size 6 12 Pack','12pk',1,'https://www.woolworths.co.nz/shop/product-details/52216/farmer-brown-eggs-cage-free-barn-size-6','exact_reference','Observed verified-store test SKU; exact reference match only.'), ('woolworths','336101',NULL,'Woolworths Eggs Barn Size 7 12 Pack','12pk',1,'https://www.woolworths.co.nz/shop/product-details/336101/woolworths-eggs-barn-size-7','pending','Observed verified-store test SKU; exact reference match only.'), ('woolworths','820276',NULL,'Henergy Eggs Cage Free Size 7 18 Pack','18pk',1,'https://www.woolworths.co.nz/shop/product-details/820276/henergy-eggs-cage-free-size-7','pending','Observed verified-store test SKU; exact reference match only.') AS v(source,source_product_id,product_id,original_name,original_specification,category_id,product_url,mapping_status,mapping_note)
       ) s
    ON t.source=s.source
   AND t.source_product_id=s.source_product_id WHEN MATCHED
   AND t.mapping_status='pending'
   AND s.product_id IS NOT NULL THEN UPDATE
   SET t.product_id=s.product_id,
       t.mapping_status=s.mapping_status,
       t.mapping_note=s.mapping_note WHEN NOT MATCHED THEN INSERT *
""")


## 06 · 补充门店维度

补入新出现的门店，保留已维护的名称、位置等信息。


In [ ]:
spark.sql(r"""
MERGE INTO workspace.default.silver_dim_store t USING (
        SELECT DISTINCT concat(source, ':', store_id) AS store_key,
               source,
               store_id,
               source AS brand_id,
               'NZ' AS country_code
          FROM workspace.default.bronze_prices
       ) s
    ON t.store_key = s.store_key WHEN NOT MATCHED THEN INSERT (store_key,source,store_id,brand_id,country_code,location_verified)
VALUES (s.store_key,s.source,s.store_id,s.brand_id,s.country_code,false)
""")


## 07 · 补充日期维度

为报价日期生成日期维度，已有日期不重复插入。


In [ ]:
spark.sql(r"""
MERGE INTO workspace.default.silver_dim_date t USING (
        SELECT DISTINCT cast(date_format(collected_date,'yyyyMMdd') AS INT) date_id,
               collected_date calendar_date,
               year(collected_date) year,
               quarter(collected_date) quarter,
               month(collected_date)
         month,
               day(collected_date) day,
               weekday(collected_date)+1 day_of_week,
               weekday(collected_date)>=5 is_weekend
          FROM workspace.default.bronze_prices
       ) s
    ON t.date_id=s.date_id WHEN NOT MATCHED THEN INSERT *
""")


## 08 · 清洗规格与计价单位

解析包装规格、识别变重商品、统一每 kg/L/个单价，同时保留原始字段。


In [ ]:
spark.sql(r"""
CREATE OR REPLACE VIEW workspace.default.silver_v_prices_clean AS
WITH parsed AS (
        SELECT *,
               lower(regexp_replace(trim(coalesce(total_quant,'')), '\\s+', '')) raw_spec_key,
               lower(trim(coalesce(unit,''))) raw_unit_key,
               nullif(regexp_extract(lower(original_name), '(\\d+(?:\\.\\d+)?\\s*(?:ml|kg|pack|pk|g|l|ea))(?:\\s+(?:bottle|carton))?\\s*$',1),'') name_spec,
               lower(original_name) RLIKE '(min\\s+order|\\d+(?:\\.\\d+)?\\s*[-–]\\s*\\d+(?:\\.\\d+)?\\s*(kg|g))' variable_weight
          FROM workspace.default.bronze_prices
         WHERE collected_date >= DATE '2026-10-27'
       ),
       normalized AS (
        SELECT *,
               CASE WHEN variable_weight                                                     THEN NULL
                    WHEN raw_spec_key RLIKE '^(\\d+(\\.\\d+)?(ml|l|g|kg|pk|pack|ea)|kg|ea)$' THEN replace(raw_spec_key,'pack','pk')
                    ELSE replace(regexp_replace(name_spec,'\\s+',''),'pack','pk')
                     END clean_spec,
               CASE WHEN raw_unit_key RLIKE '^kg\\s*\\$'                THEN 'kg'
                    WHEN raw_unit_key IN ('kg','l','ea','100g','100ml') THEN raw_unit_key
                     END clean_unit
          FROM parsed
       ) SELECT * EXCEPT(total_quant,unit,raw_spec_key,raw_unit_key,name_spec,variable_weight,clean_spec,clean_unit),
       total_quant raw_specification,
       unit raw_reported_unit,
       clean_spec total_quant,
       clean_unit unit,
       CASE WHEN variable_weight           THEN 'variable_weight'
            WHEN clean_spec IS NULL        THEN 'unknown'
            WHEN raw_spec_key IN ('','na') THEN 'parsed_from_name'
            ELSE 'reported'
             END specification_quality_status,
       CASE WHEN clean_unit IS NULL THEN 'unknown'
            ELSE 'ok'
             END unit_quality_status,
       CASE WHEN clean_unit IN ('kg','100g') THEN 'kg'
            WHEN clean_unit IN ('l','100ml') THEN 'l'
            WHEN clean_unit='ea'             THEN 'ea'
             END normalized_unit,
       CASE WHEN clean_unit IS NOT NULL AND try_cast(unit_price AS DECIMAL(16,6))>0 THEN try_cast(unit_price AS DECIMAL(16,6))*CASE WHEN clean_unit IN ('100g','100ml') THEN 10 ELSE 1 END
             END normalized_unit_price,
       CASE WHEN clean_unit IS NOT NULL AND try_cast(member_unit_price AS DECIMAL(16,6))>0 THEN try_cast(member_unit_price AS DECIMAL(16,6))*CASE WHEN clean_unit IN ('100g','100ml') THEN 10 ELSE 1 END
             END normalized_member_unit_price
  FROM normalized
""")


## 09 · 维护商品 SKU 映射

按品牌及官网 SKU 维护映射；已确认的映射与待审核状态分别处理。


In [ ]:
spark.sql(r"""
MERGE INTO workspace.default.silver_dim_product_source t USING (
        WITH alias_map AS (
                SELECT a.source,
                       lower(regexp_replace(trim(coalesce(a.original_name,'')), '\\s+', ' ')) name_key,
                       replace(lower(regexp_replace(trim(coalesce(a.original_specification,'')), '\\s+', '')), 'pack', 'pk') spec_key,
                       p.category_id,
                       min(a.product_id) product_id
                  FROM workspace.default.silver_dim_product_alias a
                  JOIN workspace.default.silver_dim_product p
                    ON a.product_id=p.product_id
                 GROUP BY a.source,
                          lower(regexp_replace(trim(coalesce(a.original_name,'')), '\\s+', ' ')),
                          replace(lower(regexp_replace(trim(coalesce(a.original_specification,'')), '\\s+', '')), 'pack', 'pk'),
                          p.category_id
                HAVING count(DISTINCT a.product_id)=1
               ),
               candidates AS (
                SELECT b.source,
                       b.source_product_id,
                       min(a.product_id) product_id,
                       max(b.original_name) original_name,
                       max(b.total_quant) original_specification,
                       max(try_cast(b.category_id AS INT)) category_id,
                       max(b.product_url) product_url
                  FROM workspace.default.silver_v_prices_clean b
                  LEFT JOIN alias_map a
                    ON b.source=a.source
                   AND lower(regexp_replace(trim(coalesce(b.original_name,'')), '\\s+', ' '))=a.name_key
                   AND replace(lower(regexp_replace(trim(coalesce(b.total_quant,'')), '\\s+', '')), 'pack', 'pk')=a.spec_key
                   AND try_cast(b.category_id AS INT)=a.category_id
                 WHERE b.source_product_id IS NOT NULL
                   AND trim(b.source_product_id)<>''
                 GROUP BY b.source,
                          b.source_product_id
                HAVING count(DISTINCT a.product_id)<=1
               ) SELECT *,
               CASE WHEN product_id IS NOT NULL THEN 'exact_reference'
                    ELSE 'pending'
                     END mapping_status,
               'Exact name/spec/category reference match; no fuzzy matching' mapping_note
          FROM candidates
       ) s
    ON t.source=s.source
   AND t.source_product_id=s.source_product_id WHEN MATCHED
   AND t.mapping_status='pending'
   AND s.product_id IS NOT NULL THEN UPDATE
   SET t.product_id=s.product_id,
       t.mapping_status=s.mapping_status,
       t.mapping_note=s.mapping_note WHEN NOT MATCHED THEN INSERT *
""")


## 10 · 应用已核实的 SKU 映射

根据已核实的品牌、规格和商品身份补充映射。


In [ ]:
spark.sql(r"""
MERGE INTO workspace.default.silver_dim_product_source t USING (
        SELECT 'woolworths' source,
               v.*,
               max(b.product_url) product_url
          FROM (
                SELECT *
                  FROM
                VALUES ('52242','0003','Farmer Brown Eggs Cage Free Barn Size 7 12 Pack','12pk',1),('774152','0011','Woodland Eggs Free Range Size 6 10 Pack','10pk',1),('774153','0014','Woodland Eggs Free Range Size 7 10 Pack','10pk',1),('546293','0019','Meadow Fresh Milk Standard Original 1L Bottle','1l',2),('282793','0020','Meadow Fresh Milk Standard Original 2L Bottle','2l',2),('701971','0021','Meadow Fresh Milk Standard Original 3L Bottle','3l',2),('546321','0022','Meadow Fresh Calci-Trim Milk Trim Calci-Trim 1L Bottle','1l',2),('282729','0023','Meadow Fresh Calci-Trim Milk Trim Calci-Trim 2L Bottle','2l',2) AS v(source_product_id,product_id,original_name,original_specification,category_id)
               ) v
          JOIN workspace.default.silver_v_prices_clean b
            ON b.source='woolworths'
           AND b.source_product_id=v.source_product_id
           AND b.original_name=v.original_name
           AND b.total_quant=v.original_specification
           AND try_cast(b.category_id AS INT)=v.category_id
         GROUP BY ALL
       ) s
    ON t.source=s.source
   AND t.source_product_id=s.source_product_id WHEN MATCHED
   AND t.mapping_status IN ('pending','exact_reference') THEN UPDATE
   SET t.product_id=s.product_id,
       t.mapping_status='approved',
       t.original_specification=s.original_specification,
       t.mapping_note='Reviewed exact source SKU, brand, product line, egg grade/farming method or fresh bottle volume; 2026-10-27' WHEN NOT MATCHED THEN INSERT (source,source_product_id,product_id,original_name,original_specification,category_id,product_url,mapping_status,mapping_note)
VALUES (s.source,s.source_product_id,s.product_id,s.original_name,s.original_specification,s.category_id,s.product_url,'approved','Reviewed exact source card identity; 2026-10-27')
""")


## 11 · 隔离存在歧义的 SKU

暂缓映射两种身份尚未确认的 1L 牛奶 SKU，避免错误合并。


In [ ]:
spark.sql(r"""
UPDATE workspace.default.silver_dim_product_source
   SET product_id=NULL,
       mapping_status='pending_review',
       mapping_note='Two distinct source SKUs share Original Fresh Milk 1L title. Review bottle/carton/product identity before comparison; 2026-10-27'
 WHERE source IN ('newworld','paknsave')
   AND source_product_id IN ('product-5026254-EA-000','product-5026255-EA-000')
   AND mapping_status IN ('pending','exact_reference')
""")


## 12 · silver_dim_product · UPDATE

执行下方 SQL，按 Run all 的顺序处理。


In [ ]:
spark.sql(r"""
UPDATE workspace.default.silver_dim_product
   SET comparison_unit='l'
 WHERE category_id=2
   AND is_analysis_selected=true
   AND comparison_unit<>'l'
""")


## 13 · 写入 Silver 价格事实表

按门店、日期与来源 SKU 去重，关联标准商品，原子覆盖生产期事实数据。


In [ ]:
spark.sql(r"""
INSERT OVERWRITE TABLE workspace.default.silver_fact_product_price (observation_id,date_id,collected_date,store_key,product_id,category_id,source,store_id,original_name,original_specification,total_price,reported_unit_price,reported_unit,is_promo,product_mapping_status,price_quality_status,source_file,ingested_at,source_product_id,product_url,member_price,member_unit_price,was_price,store_verification_status,raw_specification,raw_reported_unit,specification_quality_status,unit_quality_status,normalized_unit,normalized_unit_price,normalized_member_unit_price) WITH alias_map AS (
        SELECT a.source,
               lower(regexp_replace(trim(coalesce(a.original_name,'')), '\\s+', ' ')) name_key,
               replace(lower(regexp_replace(trim(coalesce(a.original_specification,'')), '\\s+', '')), 'pack', 'pk') spec_key,
               p.category_id,
               min(a.product_id) product_id
          FROM workspace.default.silver_dim_product_alias a
          JOIN workspace.default.silver_dim_product p
            ON a.product_id=p.product_id
         GROUP BY a.source,
                  lower(regexp_replace(trim(coalesce(a.original_name,'')), '\\s+', ' ')),
                  replace(lower(regexp_replace(trim(coalesce(a.original_specification,'')), '\\s+', '')), 'pack', 'pk'),
                  p.category_id
        HAVING count(DISTINCT a.product_id)=1
       ),
       ranked AS (
        SELECT *,
               row_number() OVER (PARTITION BY source,store_id,collected_date,category_id, CASE WHEN nullif(source_product_id,'') IS NOT NULL THEN source_product_id ELSE concat_ws('||',original_name,coalesce(total_quant,'')) END ORDER BY ingested_at DESC,source_file DESC,total_price DESC,unit_price DESC,is_promo DESC) rn
          FROM workspace.default.silver_v_prices_clean
         WHERE collected_date >= DATE '2026-10-27'
       ),
       enriched AS (
        SELECT b.*,
               p.product_id mapped_product_id,
               coalesce(p.category_id,a.category_id,c.category_id) mapped_category_id,
               CASE WHEN p.product_id IS NOT NULL          THEN 'source_id'
                    WHEN s.mapping_status='pending_review' THEN 'pending_review'
                    ELSE 'unmapped'
                     END mapping_method,
               try_cast(b.total_price AS DECIMAL(12,4)) price_value,
               try_cast(b.unit_price AS DECIMAL(12,4)) unit_price_value
          FROM ranked b
          LEFT JOIN workspace.default.silver_dim_product_source s
            ON b.source=s.source
           AND b.source_product_id=s.source_product_id
          LEFT JOIN workspace.default.silver_dim_product p
            ON s.product_id=p.product_id
           AND s.mapping_status IN ('approved','exact_reference')
           AND p.category_id=try_cast(b.category_id AS INT)
           AND replace(lower(regexp_replace(trim(coalesce(p.specification,'')), '\\s+', '')), 'pack', 'pk')=replace(lower(regexp_replace(trim(coalesce(b.total_quant,'')), '\\s+', '')), 'pack', 'pk')
          LEFT JOIN alias_map a
            ON b.source=a.source
           AND lower(regexp_replace(trim(coalesce(b.original_name,'')), '\\s+', ' '))=a.name_key
           AND replace(lower(regexp_replace(trim(coalesce(b.total_quant,'')), '\\s+', '')), 'pack', 'pk')=a.spec_key
           AND try_cast(b.category_id AS INT)=a.category_id
          LEFT JOIN workspace.default.silver_dim_category c
            ON try_cast(b.category_id AS INT)=c.category_id
         WHERE b.rn=1
       ) SELECT sha2(to_json(named_struct('source',source,'store_id',store_id,'date',collected_date,'category',category_id, 'identity',CASE WHEN nullif(source_product_id,'') IS NOT NULL THEN source_product_id ELSE concat_ws('||',original_name,coalesce(total_quant,'')) END)),256),
       cast(date_format(collected_date,'yyyyMMdd') AS INT),
       collected_date,
       concat(source,':',store_id),
       mapped_product_id,
       mapped_category_id,
       source,
       store_id,
       original_name,
       total_quant,
       CASE WHEN price_value>=0 THEN price_value
             END,
       CASE WHEN unit_price_value>=0 THEN unit_price_value
             END,
       unit,
       CASE WHEN is_promo='1' THEN true
            WHEN is_promo='0' THEN false
             END,
       mapping_method,
       CASE WHEN price_value IS NULL THEN 'missing_or_invalid_price'
            WHEN price_value<0       THEN 'negative_price'
            WHEN price_value=0       THEN 'zero_price'
            ELSE 'ok'
             END,
       source_file,
       ingested_at,
       source_product_id,
       product_url,
       CASE WHEN try_cast(member_price AS DECIMAL(12,4))>=0 THEN try_cast(member_price AS DECIMAL(12,4))
             END,
       CASE WHEN try_cast(member_unit_price AS DECIMAL(12,4))>=0 THEN try_cast(member_unit_price AS DECIMAL(12,4))
             END,
       CASE WHEN try_cast(was_price AS DECIMAL(12,4))>=0 THEN try_cast(was_price AS DECIMAL(12,4))
             END,
       CASE WHEN try_cast(store_verified AS BOOLEAN)=true THEN 'verified'
            ELSE 'legacy_unverified'
             END,
       raw_specification,
       raw_reported_unit,
       specification_quality_status,
       unit_quality_status,
       normalized_unit,
       normalized_unit_price,
       normalized_member_unit_price
  FROM enriched
""")


## 14 · 分析商品清单

列出已选定的肉、蛋、奶分析商品与映射覆盖。


In [ ]:
spark.sql(r"""
CREATE OR REPLACE VIEW workspace.default.gold_v_analysis_products AS SELECT p.*,
       c.category_name,
       (
        SELECT count(DISTINCT s.source)
          FROM workspace.default.silver_dim_product_source s
         WHERE s.product_id=p.product_id
           AND s.mapping_status IN ('approved','exact_reference')
       ) source_id_brand_coverage
  FROM workspace.default.silver_dim_product p
  JOIN workspace.default.silver_dim_category c
    ON p.category_id=c.category_id
 WHERE p.is_analysis_selected=true
""")


## 15 · 生成可比报价视图

只选择分析商品，并根据价格、规格、单位和映射状态判断是否可比。


In [ ]:
spark.sql(r"""
CREATE OR REPLACE VIEW workspace.default.gold_v_analysis_prices AS
WITH candidates AS (
        SELECT f.*,
               p.product_name,
               p.specification standard_specification,
               p.comparison_unit,
               p.analysis_rank,
               p.comparison_ready product_comparison_ready,
               count(*) OVER (PARTITION BY f.collected_date,f.store_key,f.product_id) mapped_skus_in_store
          FROM workspace.default.silver_fact_product_price f
          JOIN workspace.default.silver_dim_product p
            ON f.product_id=p.product_id
         WHERE p.is_analysis_selected=true
       ) SELECT * EXCEPT(product_comparison_ready),
       coalesce(product_comparison_ready AND store_verification_status='verified' AND price_quality_status='ok' AND unit_quality_status='ok' AND normalized_unit_price>0 AND mapped_skus_in_store=1 AND specification_quality_status IN ('reported','parsed_from_name') AND product_mapping_status IN ('source_id','name_spec_exact'),false) comparison_ready
  FROM candidates
""")


## 16 · 未映射商品汇总

保留尚未关联标准商品的报价，便于后续补映射。


In [ ]:
spark.sql(r"""
CREATE OR REPLACE VIEW workspace.default.gold_v_unmapped_products AS SELECT source,
       source_product_id,
       category_id,
       original_name,
       original_specification,
       count(*) observations,
       min(collected_date) first_seen,
       max(collected_date) last_seen
  FROM workspace.default.silver_fact_product_price
 WHERE product_id IS NULL
 GROUP BY source,
          source_product_id,
          category_id,
          original_name,
          original_specification
""")


## 17 · 生成待映射清单

按来源 SKU 汇总待审核商品、规范规格、官网链接与审核原因。


In [ ]:
spark.sql(r"""
CREATE OR REPLACE VIEW workspace.default.silver_v_product_mapping_queue AS SELECT f.source,
       f.source_product_id,
       f.category_id,
       max(f.original_name) original_name,
       max(f.original_specification) normalized_specification,
       max(f.raw_specification) raw_specification,
       max(f.product_url) product_url,
       max(s.mapping_status) registry_status,
       max(s.mapping_note) review_note,
       count(*) observations,
       count(DISTINCT f.store_key) stores,
       min(f.collected_date) first_seen,
       max(f.collected_date) last_seen
  FROM workspace.default.silver_fact_product_price f
  LEFT JOIN workspace.default.silver_dim_product_source s
    ON f.source=s.source
   AND f.source_product_id=s.source_product_id
 WHERE f.product_id IS NULL
 GROUP BY f.source,
          f.source_product_id,
          f.category_id
""")


## 18 · 生成数据质量汇总

按日期、品牌和类别统计缺价、未知单位、规格和映射情况。


In [ ]:
spark.sql(r"""
CREATE OR REPLACE VIEW workspace.default.silver_v_price_quality AS SELECT collected_date,
       source,
       category_id,
       count(*) observations,
       count(DISTINCT store_key) stores,
       count_if(product_id IS NULL) unmapped,
       count_if(price_quality_status<>'ok') invalid_price,
       count_if(unit_quality_status<>'ok') unknown_unit,
       count_if(specification_quality_status='unknown') unknown_specification,
       count_if(specification_quality_status='variable_weight') variable_weight
  FROM workspace.default.silver_fact_product_price
 GROUP BY collected_date,
          source,
          category_id
""")


## 检查结果

查看处理结果及质量检查；运行失败时先查看对应步骤的报错。


In [ ]:
display(spark.sql("SELECT * FROM workspace.default.gold_v_analysis_products ORDER BY category_id,analysis_rank"))
